In [2]:
import pandas as pd
import re

Carregando o mapa de sintomas


In [9]:
mapa = pd.read_csv("../data/sintomas_doencas.csv")

mapa

,Sintoma 1,Sintoma 2,Doença Associada
0,dor no peito,pressão no peito,Infarto
1,dor no peito,aperto no tórax,Infarto
2,pressão no peito,desconforto no peito,Infarto
3,pressão forte no peito,pressão no peito,Infarto
4,cansaço,fadiga,Insuficiência Cardíaca
5,cansaço,falta de energia,Insuficiência Cardíaca
6,falta de ar,dificuldade para respirar,Angina
7,falta de ar,respiração difícil,Angina
8,palpitação,palpitações,Arritmia
9,palpitação,coração acelerado,Arritmia


Agora realizar a leitura

In [10]:
with open("../textos/frases_sintomas.txt", "r", encoding="utf-8") as arquivo:
    frases = arquivo.readlines()

frases

['Há dois dias estou sentindo uma pressão forte no peito que piora quando subo escadas e preciso parar para descansar.\n',
 '\n',
 'Estou com muito cansaço há cerca de uma semana e tenho me sentido sem energia mesmo depois de uma noite inteira de sono.\n',
 '\n',
 'Desde ontem sinto falta de ar quando caminho por alguns minutos e preciso diminuir o ritmo para conseguir respirar melhor.\n',
 '\n',
 'Nas últimas três noites percebi que meu coração fica acelerado e tenho palpitações mesmo quando estou parado.\n',
 '\n',
 'Há aproximadamente cinco dias estou sentindo tontura e fraqueza, principalmente quando me levanto rapidamente.\n',
 '\n',
 'Estou com inchaço nas pernas há duas semanas e percebi que minhas pernas ficam mais pesadas no final do dia.\n',
 '\n',
 'Desde esta manhã sinto uma dor no peito que aparece quando faço esforço físico e melhora quando fico em repouso.\n',
 '\n',
 'Há três dias estou sentindo dificuldade para respirar durante a noite e acordo algumas vezes com falta 

Verificando quantas frases existem 

In [11]:
print("Quantidade de frases:", len(frases))

Quantidade de frases: 19


Procurando os sintomas dentro das frases

In [12]:
sintomas = []

for coluna in ["Sintoma 1", "Sintoma 2"]:
    sintomas.extend(mapa[coluna].dropna().tolist())

sintomas = list(set(sintomas))

print(sintomas)

['dor leve nas costas', 'palpitação', 'palpitações', 'perda de energia', 'falta de ar', 'tontura', 'aperto no tórax', 'pressão no peito', 'coração acelerado', 'pernas inchadas', 'dificuldade para respirar', 'fraqueza', 'desconforto nas costas', 'falta de energia', 'pressão forte no peito', 'dor no peito', 'respiração difícil', 'dor nas costas', 'cansaço', 'inchaço nas pernas', 'fadiga', 'desconforto no peito', 'vertigem']


Analisando as frases

In [13]:
for frase in frases:
    frase = frase.strip().lower()

    # Ignora linhas vazias
    if not frase:
        continue

    sintomas_encontrados = []

    for sintoma in sintomas:
        if sintoma.lower() in frase:
            sintomas_encontrados.append(sintoma)

    print("Frase:", frase)
    print("Sintomas encontrados:", sintomas_encontrados)
    print("-" * 60)

Frase: há dois dias estou sentindo uma pressão forte no peito que piora quando subo escadas e preciso parar para descansar.
Sintomas encontrados: ['pressão forte no peito']
------------------------------------------------------------
Frase: estou com muito cansaço há cerca de uma semana e tenho me sentido sem energia mesmo depois de uma noite inteira de sono.
Sintomas encontrados: ['cansaço']
------------------------------------------------------------
Frase: desde ontem sinto falta de ar quando caminho por alguns minutos e preciso diminuir o ritmo para conseguir respirar melhor.
Sintomas encontrados: ['falta de ar']
------------------------------------------------------------
Frase: nas últimas três noites percebi que meu coração fica acelerado e tenho palpitações mesmo quando estou parado.
Sintomas encontrados: ['palpitações']
------------------------------------------------------------
Frase: há aproximadamente cinco dias estou sentindo tontura e fraqueza, principalmente quando me l

Melhorando o fluxo de consultas de dados

In [14]:
for frase in frases:
    frase = frase.strip().lower()

    if not frase:
        continue

    sintomas_encontrados = []

    for sintoma in sintomas:
        if sintoma.lower() in frase:
            sintomas_encontrados.append(sintoma)

    print("Frase:", frase)
    print("Sintomas encontrados:", sintomas_encontrados)

    doencas = []

    for sintoma in sintomas_encontrados:
        resultados = mapa[
            (mapa["Sintoma 1"].str.lower() == sintoma.lower()) |
            (mapa["Sintoma 2"].str.lower() == sintoma.lower())
        ]

        for doenca in resultados["Doença Associada"]:
            if doenca not in doencas:
                doencas.append(doenca)

    print("Possíveis associações:", doencas)
    print("-" * 70)

Frase: há dois dias estou sentindo uma pressão forte no peito que piora quando subo escadas e preciso parar para descansar.
Sintomas encontrados: ['pressão forte no peito']
Possíveis associações: ['Infarto']
----------------------------------------------------------------------
Frase: estou com muito cansaço há cerca de uma semana e tenho me sentido sem energia mesmo depois de uma noite inteira de sono.
Sintomas encontrados: ['cansaço']
Possíveis associações: ['Insuficiência Cardíaca']
----------------------------------------------------------------------
Frase: desde ontem sinto falta de ar quando caminho por alguns minutos e preciso diminuir o ritmo para conseguir respirar melhor.
Sintomas encontrados: ['falta de ar']
Possíveis associações: ['Angina']
----------------------------------------------------------------------
Frase: nas últimas três noites percebi que meu coração fica acelerado e tenho palpitações mesmo quando estou parado.
Sintomas encontrados: ['palpitações']
Possíveis 

PARTE 2 - Treinando o modelo

In [17]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
%pip install scikit-learn

Carregar o csv

In [18]:
dados_risco = pd.read_csv("../data/frases_risco.csv")

dados_risco

,frase,situacao
0,sinto uma dor forte no peito e falta de ar,alto risco
1,estou com uma pressão intensa no peito há algu...,alto risco
2,tenho dificuldade para respirar e meu coração ...,alto risco
3,sinto dor no peito quando faço esforço físico,alto risco
4,estou com falta de ar intensa e muita fraqueza,alto risco
5,acordei durante a noite com falta de ar e pres...,alto risco
6,tenho palpitações frequentes acompanhadas de t...,alto risco
7,sinto um aperto forte no peito que não melhora...,alto risco
8,estou com um leve incômodo nas costas depois d...,baixo risco
9,tenho uma dor leve no pescoço desde ontem,baixo risco


Separando as frases e as classificações

In [19]:
X = dados_risco["frase"]
y = dados_risco["situacao"]

Dividindo os dados

In [20]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

Transformando o texto em numero, utilizando o TF-IDF

In [21]:
vetorizador = TfidfVectorizer()

X_treino_tfidf = vetorizador.fit_transform(X_treino)
X_teste_tfidf = vetorizador.transform(X_teste)

In [23]:
print("Formato dos dados de treinamento:", X_treino_tfidf.shape)

Formato dos dados de treinamento: (12, 58)


Treinando o modelo

In [24]:
modelo = LogisticRegression()

modelo.fit(X_treino_tfidf, y_treino)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


Fazendo previsões no conjunto teste

In [25]:
y_pred = modelo.predict(X_teste_tfidf)

print(y_pred)

['alto risco' 'baixo risco' 'baixo risco' 'baixo risco']


Acurácia


In [26]:
acuracia = accuracy_score(y_teste, y_pred)

print("Acurácia:", acuracia)

Acurácia: 0.75


Verificando um relatório mais completo

In [27]:
print(classification_report(y_teste, y_pred))

              precision    recall  f1-score   support

  alto risco       1.00      0.50      0.67         2
 baixo risco       0.67      1.00      0.80         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4



Testando novas frases para verificar o treinamento

In [28]:
novas_frases = [
    "estou sentindo uma pressão muito forte no peito e falta de ar",
    "tenho um pequeno desconforto nas costas depois de trabalhar",
    "meu coração está acelerado e estou sentindo tontura",
    "estou com uma dor muscular leve depois de fazer exercícios"
]

novas_frases_tfidf = vetorizador.transform(novas_frases)

previsoes = modelo.predict(novas_frases_tfidf)

for frase, previsao in zip(novas_frases, previsoes):
    print("Frase:", frase)
    print("Classificação:", previsao)
    print("-" * 60)

Frase: estou sentindo uma pressão muito forte no peito e falta de ar
Classificação: alto risco
------------------------------------------------------------
Frase: tenho um pequeno desconforto nas costas depois de trabalhar
Classificação: baixo risco
------------------------------------------------------------
Frase: meu coração está acelerado e estou sentindo tontura
Classificação: alto risco
------------------------------------------------------------
Frase: estou com uma dor muscular leve depois de fazer exercícios
Classificação: baixo risco
------------------------------------------------------------


Verificando se o modelo treinado é capaz de analizar corretamente frases ambíguas avaliando o grau de risco


In [29]:
frases_ambiguas = [
    "estou com um leve aperto no peito desde ontem",
    "estou muito cansado depois de um dia inteiro de trabalho",
    "tenho uma dor no peito bem forte, mas estou me sentindo bem",
    "estou com falta de ar leve depois de subir escadas",
    "meu coração está acelerado porque fiquei nervoso",
    "tenho uma dor nas costas muito forte desde ontem"
]

frases_ambiguas_tfidf = vetorizador.transform(frases_ambiguas)

previsoes_ambiguas = modelo.predict(frases_ambiguas_tfidf)

for frase, previsao in zip(frases_ambiguas, previsoes_ambiguas):
    print("Frase:", frase)
    print("Classificação:", previsao)
    print("-" * 70)

Frase: estou com um leve aperto no peito desde ontem
Classificação: alto risco
----------------------------------------------------------------------
Frase: estou muito cansado depois de um dia inteiro de trabalho
Classificação: baixo risco
----------------------------------------------------------------------
Frase: tenho uma dor no peito bem forte, mas estou me sentindo bem
Classificação: alto risco
----------------------------------------------------------------------
Frase: estou com falta de ar leve depois de subir escadas
Classificação: alto risco
----------------------------------------------------------------------
Frase: meu coração está acelerado porque fiquei nervoso
Classificação: alto risco
----------------------------------------------------------------------
Frase: tenho uma dor nas costas muito forte desde ontem
Classificação: baixo risco
----------------------------------------------------------------------


Verificando palavras consideradas importantes pra o modelo

In [30]:
palavras = vetorizador.get_feature_names_out()
coeficientes = modelo.coef_[0]

import pandas as pd

importancias = pd.DataFrame({
    "palavra": palavras,
    "peso": coeficientes
})

importancias = importancias.sort_values("peso", ascending=False)

importancias

,palavra,peso
12,depois,0.444973
31,leve,0.388429
38,nas,0.331201
10,costas,0.331201
56,um,0.281632
36,muscular,0.257152
52,sentado,0.246802
17,dor,0.185453
21,exercícios,0.150778
14,desconforto,0.150778


In [31]:
print("Palavras mais associadas a alto risco:")
print(importancias.head(10))

print("\nPalavras mais associadas a baixo risco:")
print(importancias.tail(10))

Palavras mais associadas a alto risco:
        palavra      peso
12       depois  0.444973
31         leve  0.388429
38          nas  0.331201
10       costas  0.331201
56           um  0.281632
36     muscular  0.257152
52      sentado  0.246802
17          dor  0.185453
21   exercícios  0.150778
14  desconforto  0.150778

Palavras mais associadas a baixo risco:
     palavra      peso
8        com -0.163904
26  fraqueza -0.165258
34     muita -0.165258
47   pressão -0.231181
30   intensa -0.261092
25     forte -0.272910
22     falta -0.356412
4         ar -0.356412
39        no -0.399684
43     peito -0.399684


Comparando duas frases iguais (com sintomas pareceidos) colocadas de formas diferentes

In [32]:
comparacao = [
    "estou com dor no peito",
    "estou com uma dor leve nas costas"
]

comparacao_tfidf = vetorizador.transform(comparacao)

previsoes = modelo.predict(comparacao_tfidf)

for frase, previsao in zip(comparacao, previsoes):
    print(f"{frase} → {previsao}")

estou com dor no peito → alto risco
estou com uma dor leve nas costas → baixo risco
